In [1]:
# Leemos los datos de población del censo de 2022 a nivel de hogar y sexo
import pandas as pd
import pyarrow

url = "https://github.com/Michaeljo112/estadisticalibro/releases/download/popec22/popec22.parquet"

df = pd.read_parquet(url, engine='pyarrow')

df.head()

,I01,I02,I03,I04,I05,I10,INH,P00,P01,P02,P03
500000,1,1,67,1,1,4,1,4,3,2,26
500001,1,1,67,1,1,4,1,5,5,1,32
500002,1,1,67,1,1,4,1,6,6,2,2
500003,1,1,67,1,1,4,1,7,9,2,19
500004,1,1,67,1,1,5,1,1,1,1,70


In [2]:
# Diccionario de columnas
"""
I01 Provincia
I02 Identificador de Cantón
I03 Identificador de Parroquia
I04 Zona
I05 Sector
I10 Número de vivienda
INH Número de hogar
P00 Número de persona
P01 Parentesco o relación con el representante del hogar
P02 Sexo al nacer
P03 Años cumplidos
"""

excluir = ['P00','P01','P02','P03']

agrupacion = [c for c in df.columns if c not in excluir]

# Al obtener el número máximo del número de persona obtendremos la cantidad de personas del hogar
personasporhogar = (
    df.groupby(agrupacion)
      .agg(numpersonas=('P00','max'))
      .reset_index()
)

personasporhogar

,I01,I02,I03,I04,I05,I10,INH,numpersonas
0,1,1,50,1,1,1,1,5
1,1,1,50,1,1,2,1,4
2,1,1,50,1,1,3,1,4
3,1,1,50,1,1,4,1,4
4,1,1,50,1,1,5,1,4
...,...,...,...,...,...,...,...,...
2784857,24,3,52,999,4,70,1,3
2784858,24,3,52,999,4,71,1,2
2784859,24,3,52,999,4,72,1,1
2784860,24,3,52,999,4,73,1,1


In [3]:
# Excluye hogar que están clasificados con el número 0. Así que se respetó el filtro
pararesumen = personasporhogar[personasporhogar['INH']>0]

# Ver un conteo de la cantdiad de personas en el hogar
resumen = pararesumen['numpersonas'].value_counts().sort_index().reset_index(name='numhogares')

resumen

,numpersonas,numhogares
0,1,314250
1,2,431560
2,3,536947
3,4,633787
4,5,451793
5,6,224034
6,7,100970
7,8,45275
8,9,21191
9,10,10395


In [4]:
import altair as alt

chart = alt.Chart(resumen).mark_bar(
  size=20 # controla el grosor de las barras
).encode(
  x=alt.X(
    'numpersonas:O',
    title='Número de personas en el hogar',
    axis=alt.Axis(
      labelAngle=0, # etiquetas horizontales
    )
  ),
  y=alt.Y(
    'numhogares:Q',
    title='Número de hogares',
    axis=alt.Axis(format='.0s')
  ),
  tooltip=['numpersonas','numhogares']
).properties(
  title="Distribución del tamaño del hogar",
  width=700, # longitud eje horizontal 
  height=400
)

chart

alt.Chart(...)

In [5]:
hogarestotales = sum(resumen['numhogares'])
print(f'Hay {hogarestotales} hogares en total.')
resumen['porcentaje'] = resumen['numhogares']/hogarestotales

# configurar por defecto la visualización de decimales (5 dígitos)
pd.options.display.float_format = '{:.5f}'.format

resumen

Hay 2780237 hogares en total.


,numpersonas,numhogares,porcentaje
0,1,314250,0.11303
1,2,431560,0.15522
2,3,536947,0.19313
3,4,633787,0.22796
4,5,451793,0.16250
5,6,224034,0.08058
6,7,100970,0.03632
7,8,45275,0.01628
8,9,21191,0.00762
9,10,10395,0.00374


In [6]:
chart = alt.Chart(resumen).mark_bar(
    size=20   # controla el grosor de las barras
).encode(
  x=alt.X(
    'numpersonas:O',
    title='Número de personas en el hogar',
    axis=alt.Axis(labelAngle=0)
  ),
  y=alt.Y(
    'porcentaje:Q',
    title='Frecuencia relativa',
    axis=alt.Axis(format='.0s') # compacto K
  ),
  tooltip=['porcentaje','numhogares']
).properties(
  title="Distribución relativa del tamaño del hogar",
  width=700, 
  height=400
)

chart

alt.Chart(...)